### Cells used for the construction of the final code

In [ ]:
import subprocess
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import re
import csv
import shutil


In [ ]:
def find_and_load_dlc_file(base_dir, mouse_id, target_date):
    """
    Searches only TMaze sessions (ignoring Linear) for a DLC CSV file matching the given mouse and date.

    Parameters:
        base_dir (str): Base path to AgingMiceNWB
        mouse_id (str): e.g., 'Mouse1637'
        target_date (str): e.g., '2024-06-20'

    Returns:
        pd.DataFrame or None: Loaded DLC data or None if not found
    """
    mouse_path = os.path.join(base_dir, f"sub-{mouse_id}")
    
    for root, dirs, files in os.walk(mouse_path):
        # Filter only TMaze directories and exclude Linear
        if "/TMaze/" not in root or "/Linear/" in root:
            continue

        for file in files:
            if (
                file.endswith(".csv") and
                file.startswith(f"behavior{target_date}") and
                "DLC_resnet50_T-Maze" in file
            ):
                full_path = os.path.join(root, file)
                print(f"[FOUND] Loading DLC file:\n{full_path}")
                return pd.read_csv(full_path, header = [0,1,2])
    
    print(f"[WARNING] No TMaze DLC file found for mouse {mouse_id} on {target_date}")
    return None


In [ ]:
base_directory = "/media/toor/SeagatePortableDrive/AgingMiceNWB"
mouse = "Mouse1637"
date = "2024-06-10"

dlc_data = find_and_load_dlc_file(base_directory, mouse, date)

if dlc_data is not None:
    display(dlc_data.head())


In [ ]:
# Step 1: Get all columns with 'likelihood' in the 3rd level of the multi-index
likelihood_cols = [col for col in dlc_data.columns if len(col) == 3 and col[2].lower() == 'likelihood']

# Step 2: Compute consistency score for each bodypart
likelihood_consistency = {}

for col in likelihood_cols:
    bodypart = col[1]
    series = pd.to_numeric(dlc_data[col], errors='coerce')
    total = series.notna().sum()
    high_conf = (series > 0.8).sum()
    
    if total > 0:
        score = high_conf / total
        likelihood_consistency[bodypart] = score

# Step 3: Find bodypart with highest consistency
if likelihood_consistency:
    most_consistent = max(likelihood_consistency, key=likelihood_consistency.get)
    print(f"Most consistently high-confidence bodypart: {most_consistent}")
else:
    print("⚠️ No likelihood data found.")


In [ ]:
# Step 1: Identify all 'likelihood' columns in the 3rd index
likelihood_cols = [
    col for col in dlc_data.columns
    if len(col) == 3 and col[2].lower() == 'likelihood'
]

# Step 2: Score bodyparts by % of high-confidence frames
likelihood_consistency = {}
for col in likelihood_cols:
    bodypart = col[1]
    series = pd.to_numeric(dlc_data[col], errors='coerce')
    total = series.notna().sum()
    high_conf = (series > 0.9).sum()
    if total > 0:
        score = high_conf / total
        likelihood_consistency[bodypart] = score

# Step 3: Choose the most consistently confident bodypart
if not likelihood_consistency:
    raise ValueError("No consistent bodyparts found with likelihood data.")
most_consistent_bp = max(likelihood_consistency, key=likelihood_consistency.get)

# Step 4: Extract x, y, and likelihood for that bodypart
x_col = [col for col in dlc_data.columns if col[1] == most_consistent_bp and col[2] == 'x'][0]
y_col = [col for col in dlc_data.columns if col[1] == most_consistent_bp and col[2] == 'y'][0]
l_col = [col for col in dlc_data.columns if col[1] == most_consistent_bp and col[2].lower() == 'likelihood'][0]

x = pd.to_numeric(dlc_data[x_col], errors='coerce')
y = pd.to_numeric(dlc_data[y_col], errors='coerce')
likelihood = pd.to_numeric(dlc_data[l_col], errors='coerce')
frame_idx = dlc_data.index.astype(float)

# Step 5: Mask out low-likelihood values
threshold = 0.9
low_likelihood_mask = likelihood < threshold
x[low_likelihood_mask] = None
y[low_likelihood_mask] = None

# Step 6: Interpolate missing values
x = x.interpolate()
y = y.interpolate()

# Step 7: Display summary
print(f"📌 Bodypart: {most_consistent_bp}")
print(f"Total frames: {len(likelihood)}")
print(f"Low-likelihood frames: {low_likelihood_mask.sum()}")

# Step 8: Plot Y coordinate over time
fig = go.Figure()
fig.add_trace(go.Scatter(y=y, mode='lines', name=f'{most_consistent_bp} Y'))

fig.update_layout(
    title=f'{most_consistent_bp} Y Coordinate Over Time (Confidence ≥ {threshold})',
    xaxis_title='Frame Index',
    yaxis_title='Y Coordinate'
)

fig.show()


In [ ]:
# 1) See exactly which bodypart we selected and its Y stats
print("Chosen bodypart:", most_consistent_bp)

y_cols = [c for c in dlc_data.columns if len(c)==3 and c[1]==most_consistent_bp and c[2].strip().lower()=='y']
if not y_cols:
    raise RuntimeError("Couldn't find a Y column for the chosen bodypart. Check header casing/spacing.")

y_col = y_cols[0]
y = pd.to_numeric(dlc_data[y_col], errors='coerce')

print("Number NaNs:", y.isna().sum(), " / ", len(y))

# 2) Force the plot to show the full min..max range, just in case autorange is misleading
fig = go.Figure()
fig.add_trace(go.Scatter(y=y, mode='lines', name=f'{most_consistent_bp} Y (raw)'))
fig.update_layout(
    title=f'Raw Y of {most_consistent_bp}',
    xaxis_title='Frame',
    yaxis_title='Y (pixels)',
    yaxis=dict(range=[float(y.min(skipna=True)), float(y.max(skipna=True))])
)
fig.show()


#### first time surpassing the threshod y --> to use for video trimming

In [ ]:
likelihood_thresh = 0.9
TMaze_maximum_threahold = 45
min_frame = 5000
# first_surpass_index =y.index[(y.index >= min_frame) & (y > TMaze_maximum_threahold)][0] 
frame_rate = 25
# start_time_seconds = first_surpass_index/frame_rate
# start_time_seconds

In [ ]:
y.index[(y.index >= min_frame) & (y > TMaze_maximum_threahold)][0]

In [ ]:
pos_after = np.flatnonzero(mask & (np.arange(mask.size) >= min_frame))

##### Simple Video Trimming for a single directory

In [ ]:
def crossing_mask(y_series):
    return pd.to_numeric(y_series, errors='coerce') < threshold

def pick_most_robust_bodypart(dlc_df, thr=0.9):
    """Return bodypart name with highest fraction of frames having likelihood > thr."""
    # find likelihood columns in the 3rd level of the MultiIndex
    lcols = [c for c in dlc_df.columns if len(c) == 3 and str(c[2]).strip().lower() == 'likelihood']
    if not lcols:
        raise ValueError("No likelihood columns found in DLC data.")
    scores = {}
    for col in lcols:
        bp = col[1]
        s = pd.to_numeric(dlc_df[col], errors='coerce')
        total = s.notna().sum()
        if total == 0:
            continue
        score = (s > thr).sum() / total
        # keep the best score per bodypart in case of multiple scorers
        scores[bp] = max(scores.get(bp, 0.0), float(score))
    if not scores:
        raise ValueError("No valid likelihood data to compute robustness.")
    return max(scores, key=scores.get)

def col_of(dlc_df, bodypart, coord):
    """Return MultiIndex column for given bodypart and coord in {'x','y','likelihood'}."""
    cs = [c for c in dlc_df.columns if len(c) == 3
          and c[1] == bodypart
          and str(c[2]).strip().lower() == coord]
    if not cs:
        raise ValueError(f"Missing column for bodypart={bodypart}, coord={coord}.")
    return cs[0]


def extract_ts(fname: str) -> str | None:
    """
    Extract YYYY-MM-DDTHH_MM_SS from a filename.
    Example match: behavior2024-10-03T14_16_18.avi
    """
    m = re.search(r'(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2})', os.path.basename(fname))
    return m.group(1) if m else None


In [ ]:
dlc_data

In [ ]:
bp='neck'
y_col = col_of(dlc_data, bp, 'y')
dlc_data[y_col]

In [ ]:
y = pd.to_numeric(dlc_data[y_col], errors='coerce')

In [ ]:
likelihood = col_of(dlc_data, bp, 'likelihood')
likelihood

### Start Here

In [ ]:
import subprocess
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import re
import csv
import shutil

In [ ]:
def crossing_mask(y_series):
    return pd.to_numeric(y_series, errors='coerce') < threshold

def pick_most_robust_bodypart(dlc_df, thr=0.9):
    """Return bodypart name with highest fraction of frames having likelihood > thr."""
    # find likelihood columns in the 3rd level of the MultiIndex
    lcols = [c for c in dlc_df.columns if len(c) == 3 and str(c[2]).strip().lower() == 'likelihood']
    if not lcols:
        raise ValueError("No likelihood columns found in DLC data.")
    scores = {}
    for col in lcols:
        bp = col[1]
        s = pd.to_numeric(dlc_df[col], errors='coerce')
        total = s.notna().sum()
        if total == 0:
            continue
        score = (s > thr).sum() / total
        # keep the best score per bodypart in case of multiple scorers
        scores[bp] = max(scores.get(bp, 0.0), float(score))
    if not scores:
        raise ValueError("No valid likelihood data to compute robustness.")
    return max(scores, key=scores.get)

def col_of(dlc_df, bodypart, coord):
    """Return MultiIndex column for given bodypart and coord in {'x','y','likelihood'}."""
    cs = [c for c in dlc_df.columns if len(c) == 3
          and c[1] == bodypart
          and str(c[2]).strip().lower() == coord]
    if not cs:
        raise ValueError(f"Missing column for bodypart={bodypart}, coord={coord}.")
    return cs[0]


def extract_ts(fname: str) -> str | None:
    """
    Extract YYYY-MM-DDTHH_MM_SS from a filename.
    Example match: behavior2024-10-03T14_16_18.avi
    """
    m = re.search(r'(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2})', os.path.basename(fname))
    return m.group(1) if m else None


In [ ]:
likelihood_thresh = 0.9
TMaze_maximum_threahold = 45
min_frame = 5000
# first_surpass_index =y.index[(y.index >= min_frame) & (y > TMaze_maximum_threahold)][0] 
frame_rate = 25
# start_time_seconds = first_surpass_index/frame_rate
# start_time_seconds
Linear_minimum_threshold =  715 #anything bigger than this, catch it
min_frame = 5000
TMaze_maximum_threahold = 45

### TMaze

In [ ]:
# where to write trimmed videos
input_root = "/media/toor/Seagate2/Mouse1639/Round3/TMaze/"
output_root = "/media/toor/Seagate2/Mouse1639/Round3/TMaze_trimmed/"
os.makedirs(output_root, exist_ok=True)

In [ ]:
for root, dirs, files in os.walk(input_root, topdown=False):
    # # skip non-TMaze and any Linear directories
    if "/TMaze/" not in root or "/Linear/" in root:
        continue

    dlc_file = None
    miniscope_file = None
    behavior_file = None
    start_time_seconds = None

    for file in files:
        # Identify DLC (CSV, filtered)
        if file.endswith(".csv") and "filtered" in file:
            dlc_file = os.path.join(root, file)            
            dlc_data = pd.read_csv(dlc_file, header=[0, 1, 2]) # Load DLC CSV with 3 header rows (MultiIndex)           
            bp = pick_most_robust_bodypart(dlc_data, thr=likelihood_thresh) # Pick most robust bodypart by likelihood
            
            # Grab its raw Y (no thresholding or interpolation)
            y_col = col_of(dlc_data, bp, 'y')
            likelihood_col = col_of(dlc_data, bp, 'likelihood')
            y = pd.to_numeric(dlc_data[y_col], errors='coerce')
            likelihood = pd.to_numeric(dlc_data[likelihood_col], errors='coerce')

            # Find first crossing index safely

            start_frame=y.index[(y.index >= min_frame) & (y > TMaze_maximum_threahold) & (likelihood > likelihood_thresh)][0]   
            start_time_seconds = start_frame/frame_rate               


            print(f"[INFO] DLC: {dlc_file}")
            print(f"[INFO] Selected bodypart by robustness: {bp}")
            print(f"[INFO] First crossing at frame #{start_frame}"
                  f"start_time_seconds={start_time_seconds:.3f}")

            # --- look for videos ONE DIRECTORY ABOVE the DLC file folder ---
            parent_dir = os.path.dirname(root)  # the directory above the dlc folder
            try:
                parent_files = os.listdir(parent_dir)
            except FileNotFoundError:
                parent_files = []

            # Prefer a deterministic pick in case of multiple matches
            for pf in sorted(parent_files):
                if pf.endswith(".avi") and "miniscope" in pf:
                    miniscope_file = os.path.join(parent_dir, pf)
                    break

            for pf in sorted(parent_files):
                if pf.endswith(".avi") and pf.startswith("behavior") and not pf.startswith("behaviorLinear"):
                    behavior_file = os.path.join(parent_dir, pf)
                    break

    # If everything needed is present, trim
    if miniscope_file and behavior_file and dlc_file and start_time_seconds is not None:

        ts = extract_ts(behavior_file)
        output_dir = os.path.join(output_root, ts)
        os.makedirs(output_dir, exist_ok=True)
        
        # Path to the DLC folder inside this directory tree
        dlc_dir = os.path.join(root)

        # Only copy if the folder exists
        if os.path.isdir(dlc_dir):
            dest_dlc_dir = os.path.join(output_dir, "dlc")

            # Remove if already exists (shutil.copytree requires this)
            if os.path.exists(dest_dlc_dir):
                shutil.rmtree(dest_dlc_dir)

            print(f"[COPY] Copying DLC folder: {dlc_dir} -> {dest_dlc_dir}")
            shutil.copytree(dlc_dir, dest_dlc_dir)
        else:
            print(f"[WARN] No 'dlc' folder found in: {root}")

        # Save start_frame and start_time_seconds to a small CSV in the same folder
        csv_path = os.path.join(output_dir, "TMaze_trim_info.csv")
        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["start_frame", "start_time_seconds", "most_robust_bodypart"])
            writer.writerow([start_frame, f"{start_time_seconds:.6f}", bp])

        print(f"[META] Saved trim info: {csv_path}")
        print(f"[TRIM] start_time_seconds={start_time_seconds:.3f}")
        print(f"[TRIM] dlc:       {dlc_file}")
        print(f"[TRIM] miniscope: {miniscope_file}")
        print(f"[TRIM] behavior:  {behavior_file}")

        for video_file in [miniscope_file, behavior_file]:
            out_name = f"TMaze_{os.path.basename(video_file)}"
            output_file = os.path.join(output_dir, out_name)

            ffmpeg_command = [
                "ffmpeg",
                "-n",
                "-ss", f"{start_time_seconds:.3f}",
                "-i", video_file,
                "-t", "900",
                "-c:v", "copy",
                "-c:a", "copy",
                output_file
            ]
            print(f"[RUN] {' '.join(ffmpeg_command)}")
            try:
                subprocess.run(ffmpeg_command, check=True)
                print(f"[OK] Wrote: {output_file}")
            except subprocess.CalledProcessError as e:
                print(f"[ERR] Trimming failed for {video_file}: {e}")
    else:
        print(f"[MISS] Required files not all found in: {root}")
        if not miniscope_file: print("  - Miniscope .avi missing.")
        if not behavior_file:  print("  - Behavior .avi missing.")
        if not dlc_file:       print("  - DLC .csv missing.")
        if start_time_seconds is None: print("  - Crossing time not computed.")


### Linear

In [ ]:
# where to write trimmed videos
input_root = "/media/toor/T7Shield/Mouse1637/Round3/Linear"
output_root = "/media/toor/T7Shield/Mouse1637/Round3/Linear_trimmed/"
os.makedirs(output_root, exist_ok=True)

In [ ]:
for root, dirs, files in os.walk(input_root, topdown=False):
    # # skip non-TMaze and any Linear directories
    # if "/TMaze/" not in root or "/Linear/" in root:
    #     continue

    dlc_file = None
    miniscope_file = None
    behavior_file = None
    start_time_seconds = None

    for file in files:
        # Identify DLC (CSV, filtered)
        if file.endswith(".csv") and "filtered" in file:
            print(f"Found filtered DLC file: {file}")
            dlc_file = os.path.join(root, file)            
            dlc_data = pd.read_csv(dlc_file, header=[0, 1, 2]) # Load DLC CSV with 3 header rows (MultiIndex)           
            bp = pick_most_robust_bodypart(dlc_data, thr=likelihood_thresh) # Pick most robust bodypart by likelihood
            
            # Grab its raw Y (no thresholding or interpolation)
            y_col = col_of(dlc_data, bp, 'y')
            likelihood_col = col_of(dlc_data, bp, 'likelihood')
            y = pd.to_numeric(dlc_data[y_col], errors='coerce')
            likelihood = pd.to_numeric(dlc_data[likelihood_col], errors='coerce')

            # Find first crossing index safely

            start_frame=y.index[(y.index >= min_frame) & (y < Linear_minimum_threshold) & (likelihood > likelihood_thresh)][0]   
            start_time_seconds = start_frame/frame_rate               


            print(f"[INFO] DLC: {dlc_file}")
            print(f"[INFO] Selected bodypart by robustness: {bp}")
            print(f"[INFO] First crossing at frame #{start_frame}"
                  f"start_time_seconds={start_time_seconds:.3f}")

            # --- look for videos ONE DIRECTORY ABOVE the DLC file folder ---
            parent_dir = os.path.dirname(root)  # the directory above the dlc folder
            try:
                parent_files = os.listdir(parent_dir)
            except FileNotFoundError:
                parent_files = []

            # Prefer a deterministic pick in case of multiple matches
            for pf in sorted(parent_files):
                if pf.endswith(".avi") and "miniscope" in pf:
                    miniscope_file = os.path.join(parent_dir, pf)
                    break

            for pf in sorted(parent_files):
                if pf.endswith(".avi") and pf.startswith("behavior") and not pf.startswith("behaviorLinear"):
                    behavior_file = os.path.join(parent_dir, pf)
                    break

    # If everything needed is present, trim
    if miniscope_file and behavior_file and dlc_file and start_time_seconds is not None:

        ts = extract_ts(behavior_file)
        output_dir = os.path.join(output_root, ts)
        os.makedirs(output_dir, exist_ok=True)

        # Path to the DLC folder inside this directory tree
        dlc_dir = os.path.join(root)

        # Only copy if the folder exists
        if os.path.isdir(dlc_dir):
            dest_dlc_dir = os.path.join(output_dir, "dlc")

            # Remove if already exists (shutil.copytree requires this)
            if os.path.exists(dest_dlc_dir):
                shutil.rmtree(dest_dlc_dir)

            print(f"[COPY] Copying DLC folder: {dlc_dir} -> {dest_dlc_dir}")
            shutil.copytree(dlc_dir, dest_dlc_dir)
        else:
            print(f"[WARN] No 'dlc' folder found in: {root}")

        # Save start_frame and start_time_seconds to a small CSV in the same folder
        csv_path = os.path.join(output_dir, "Linear_trim_info.csv")
        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["start_frame", "start_time_seconds"])
            writer.writerow([start_frame, f"{start_time_seconds:.6f}"])

        print(f"[META] Saved trim info: {csv_path}")
        print(f"[TRIM] start_time_seconds={start_time_seconds:.3f}")
        print(f"[TRIM] dlc:       {dlc_file}")
        print(f"[TRIM] miniscope: {miniscope_file}")
        print(f"[TRIM] behavior:  {behavior_file}")

        for video_file in [miniscope_file, behavior_file]:
            out_name = f"Linear_{os.path.basename(video_file)}"
            output_file = os.path.join(output_dir, out_name)

            ffmpeg_command = [
                "ffmpeg",
                "-n",
                "-ss", f"{start_time_seconds:.3f}",
                "-i", video_file,
                "-t", "900",
                "-c:v", "copy",
                "-c:a", "copy",
                output_file
            ]
            print(f"[RUN] {' '.join(ffmpeg_command)}")
            try:
                subprocess.run(ffmpeg_command, check=True)
                print(f"[OK] Wrote: {output_file}")
            except subprocess.CalledProcessError as e:
                print(f"[ERR] Trimming failed for {video_file}: {e}")
    else:
        print(f"[MISS] Required files not all found in: {root}")
        if not miniscope_file: print("  - Miniscope .avi missing.")
        if not behavior_file:  print("  - Behavior .avi missing.")
        if not dlc_file:       print("  - DLC .csv missing.")
        if start_time_seconds is None: print("  - Crossing time not computed.")
